# UV-Vis Spectral Analysis — Polars & Dual-Style Plotting Workflow

This notebook uses [`uvvis_spectra.py`](./uvvis_spectra.py) to:
1. **Step 1 (Inspect)**: Auto-parse either CSV export format into a `polars.DataFrame`, list detected peaks ($\lambda_{\max}$) and troughs ($\lambda_{\min}$), and display an interactive Plotly plot.
2. **Step 2 (Subtract & Zero)**: Rename samples, optionally zero at a reference wavelength (e.g., 490 nm), and compute/store difference spectra ($\text{Sample A} - \text{Sample B}$).
3. **Step 3 (Concentration)**: Calculate protein concentration using $(A_{\text{target}} - A_{\text{ref}}) / \varepsilon$ (or single-wavelength $A_{\text{target}} / \varepsilon$) on either a difference spectrum or raw sample.
4. **Step 4 (Publish)**: Render minimal, grid-free publication plots (`.pdf` / `.svg` / `.png`).

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from uvvis_spectra import load_spectra, plot_interactive, plot_publication

## Step 1: Load & Inspect Spectra
Change `filepath` below to any spectrophotometer CSV file (either Full Report or Table-Only Export format).

In [ ]:
# Works automatically with both CSV formats:
# filepath = "2026-11-09-mb2xstar.csv"
filepath = "Export Data 2026-09-18-H6.csv"

ds = load_spectra(filepath)
print(f"Loaded '{ds.source_name}' (format: {ds.format_type})")
print("Samples:", ds.sample_names)

# Preview Polars DataFrame
display(ds.df.head(5))

# Detect positive peaks and negative troughs
extrema = ds.detect_extrema(prominence=0.005)
display(extrema)

# Interactive Plotly plot (marks both peaks and troughs; saves HTML)
fig_int = plot_interactive(
    ds,
    mark_peaks=True,
    mark_troughs=True,
    output_html=f"outputs/{ds.source_name}_interactive.html",
)
fig_int.show()

## Step 2: Optional Baseline Zeroing & Difference Spectrum (`Sample A - Sample B`)

In [ ]:
# Optional: Rename samples if desired
# ds.rename_samples({"H6-iii-100X": "Fe(III)", "Sample 2": "Fe(II)", "Sample 5": "CO_1"})

# Optional: Zero all spectra at 490 nm (set in_place=True to apply)
# ds.zero_spectra(reference_wl=490.0, in_place=True)

# Compute and store difference spectrum in the Polars DataFrame
diff_name = ds.compute_difference(
    minuend="Sample 5",
    subtrahend="Sample 2",
    name="CO - Fe(II) difference",
    zero_at=None,  # Or e.g. 490.0 to zero the difference spectrum at 490 nm
)

# Save updated Polars DataFrame (with difference column) to CSV
ds.export_csv(f"outputs/{ds.source_name}_with_diff.csv")

# Interactive plot showing selected raw spectra + difference spectrum
fig_diff_int = plot_interactive(
    ds,
    samples=["H6-iii-100X", "Sample 2", "Sample 5"],
    difference_samples=[diff_name],
    output_html=f"outputs/{ds.source_name}_diff_interactive.html",
)
fig_diff_int.show()

## Step 3: Protein Concentration Calculation
Supports dual-wavelength $(A_{\text{target}} - A_{\text{ref}}) / \varepsilon$ or single-wavelength $A_{\text{target}} / \varepsilon$ on any difference or raw spectrum.

In [ ]:
# Example 1: P450 CO-difference spectrum using peak in [445, 455] nm minus A_490
conc_result = ds.calculate_concentration(
    spectrum=diff_name,
    extinction_coeff=91000.0,     # M^-1 cm^-1
    target_wl=(445.0, 455.0),     # Or pass a single wavelength float like 450.0
    reference_wl=490.0,           # Set to None to skip baseline subtraction
    dilution_factor=50.0,
    pathlength_cm=1.0,
    unit="uM",
)
print(conc_result.summary())

## Step 4: Minimal Publication Plot (No Grid, Despined)

In [ ]:
fig_pub = plot_publication(
    ds,
    samples=["H6-iii-100X", "Sample 2", "Sample 5"],
    difference_samples=[diff_name],
    mark_peaks=True,
    mark_troughs=False,
    output_path=f"outputs/{ds.source_name}_publication.pdf",
    dpi=300,
)
plt.show()